In [ ]:
%matplotlib qt
import mne
import numpy as np
import os.path as op
import matplotlib.pyplot as plt

# Read sample data

In [ ]:
data_path = op.expanduser('~/data/wash-u/REST18_20_MZ_Best/')
fname_raw = '7005_4_rest1_ec.cnt'
fpath_raw = op.join(data_path, fname_raw)
raw = mne.io.read_raw_cnt(fpath_raw, eog='auto', ecg=['EKG'])
raw.info

# Visually inspect data for bad channels and spans

In [ ]:
# Visually inspect raw data and mark bad channels and span
# raw.plot(n_channels=len(raw))

In [ ]:
# Interpolate bad channel data
# raw.interpolate_bads()

# Channel locations

In [ ]:
# fig2 = raw.plot_sensors(show_names=True)
# raw.plot_sensors('3d')

# Use ICA to remove ocular and cardiac artifacts

In [ ]:
# eog_evoked = mne.preprocessing.create_eog_epochs(raw).average()
# eog_evoked.apply_baseline(baseline=(None, -0.2))
# fig6 = eog_evoked.plot_joint()

In [ ]:
# Load data into memory
raw.load_data()

# Filtering to remove slow drifts
filt_raw_ica = raw.copy().filter(l_freq=1, h_freq=None)

# Fitting the ICA solution
ica = mne.preprocessing.ICA(n_components=30, max_iter='auto', random_state=97)
ica.fit(filt_raw_ica)
ica

In [ ]:
# Plotting the ICA solution
fig7 = ica.plot_sources(raw, show_scrollbars=True)

# Visualize the scalp field distribution of ICA components
fig8 = ica.plot_components()

In [ ]:
# Additional visualization to check exclusion criteria
fig9 = ica.plot_overlay(raw, exclude=[2, 20])

In [ ]:
# Manual selection of ICA components to be excluded
ica.exclude = [2, 20]

# Reconstruction of raw data after ICA exclusion
reconst_raw = raw.copy()
ica.apply(reconst_raw)

# Visualize repair
fig10 = raw.plot(n_channels=len(raw), show_scrollbars=False)
fig11 = reconst_raw.plot(n_channels=len(raw), show_scrollbars=False)

In [ ]:
# Export preprocessed data to a new .fif file
fname_reconst_raw = fname_raw[:-4] + '_eeg.fif'
fpath_reconst_raw = op.join(data_path, '../preprocessed/', fname_reconst_raw)
reconst_raw.save(fpath_reconst_raw, overwrite=True)